different methods for classifying flausch / not flausch

In [1]:
import pandas as pd
from transformers import AutoTokenizer
from sklearn.linear_model import LogisticRegression

# for evaluation
from flauscheval import binary_eval

In [36]:
train_task1 = pd.read_json("Input/Data/train/train_task1_new.json")
test_task1 = pd.read_json("Input/Data/train/test_task1_new.json")

In [29]:
gold_test_task1 = test_task1[["document", "comment_id", "flausch"]].copy()
gold_test_task1.to_csv("Input/Data/train/gold_test_task1.csv", index=False)

based on sentiment analyzer for different version of the comments

In [4]:
def sentiment_prediction(polarity, cutoff):
    if polarity > cutoff:
        return "yes"
    else:
        return "no"

In [20]:
predictions_orig = []
predictions_spelling_corrected = []
predictions_translated = []

for i in range(len(test_task1)):
    predictions_orig.append(sentiment_prediction(test_task1.iloc[i]["sentiment_orig"], 0.2))
    predictions_spelling_corrected.append(sentiment_prediction(test_task1.iloc[i]["sentiment_spelling_corrected"], 0.2))
    predictions_translated.append(sentiment_prediction(test_task1.iloc[i]["sentiment_translated"], 0.2))

In [21]:
pred_orig = test_task1[["document", "comment_id"]].copy()
pred_orig["flausch"] = predictions_orig
pred_spelling_corrected = test_task1[["document", "comment_id"]].copy()
pred_spelling_corrected["flausch"] = predictions_spelling_corrected
pred_translated = test_task1[["document", "comment_id"]].copy()
pred_translated["flausch"] = predictions_translated

In [32]:
pred_orig.to_csv("predictions/pred_sentiments_orig_test_task1.csv", index=False)
pred_spelling_corrected.to_csv("predictions/pred_sentiments_spelling_corrected_test_task1.csv", index=False)
pred_translated.to_csv("predictions/pred_sentiments_translated_test_task1.csv", index=False)

In [42]:
binary_eval("Input/Data/train/gold_test_task1.csv", "predictions/pred_sentiments_orig_test_task1.csv")

(0.6409523809523809, 0.6446360153256705, 0.6427889207258835)

In [ ]:
binary_eval("Input/Data/train/gold_test_task1.csv", "predictions/pred_sentiments_spelling_corrected_test_task1.csv")

(0.646551724137931, 0.6490384615384616, 0.6477927063339731)

In [ ]:
binary_eval("Input/Data/train/gold_test_task1.csv", "predictions/pred_sentiments_translated_test_task1.csv")

(0.6628352490421456, 0.5704863973619126, 0.613203367301728)

In [ ]:
# all original comments, evaluation for flausch & not flausch classes

import pickle 
from sklearn.metrics import classification_report 

task1 = pd.read_csv("Input/Data/train/task1.csv")
with open("Input/Data/train/sentiments_orig.pkl", "rb") as infile: 
   sentiments_orig = pickle.load(infile)

y_pred = []
y_test = []

for i in range(len(task1)):

    flausch = task1.iloc[i]["flausch"]

    if flausch == "yes":
        y_test.append(1)
    else:
        y_test.append(0)


    sentiment = sentiments_orig[i]

    if sentiment > 0.2:
        y_pred.append(1)
    else:
        y_pred.append(0)

print(classification_report(y_test, y_pred, zero_division=0.0))

              precision    recall  f1-score   support

           0       0.86      0.86      0.86     26284
           1       0.67      0.66      0.67     10773

    accuracy                           0.81     37057
   macro avg       0.76      0.76      0.76     37057
weighted avg       0.81      0.81      0.81     37057



token frequency based approach

In [47]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report 

In [48]:
tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-german-cased")

In [69]:
token_dict_all = {}
token_dict_flausch = {}
token_dict_not_flausch = {}

for i in range(len(train_task1)):

    doc = train_task1.iloc[i]["document"]
    id = train_task1.iloc[i]["comment_id"]

    comment = train_task1[(train_task1["document"] == doc) & (train_task1["comment_id"] == id)].iloc[0]["comment"]
    # iloc[0] needed when multiple entries because of different spans from task 2

    tokens = tokenizer.tokenize(comment)
    ids = tokenizer.convert_tokens_to_ids(tokens)


    for j in ids:
        if j not in token_dict_all.keys():
            token_dict_all[j] = 0
        if j not in token_dict_flausch.keys():
            token_dict_flausch[j] = 0
        if j not in token_dict_not_flausch.keys():
            token_dict_not_flausch[j] = 0

        if task1.iloc[i]["flausch"] == "yes":
            token_dict_all[j] += 1
            token_dict_flausch[j] += 1
        else:
            token_dict_all[j] += 1
            token_dict_not_flausch[j] += 1

Token indices sequence length is longer than the specified maximum sequence length for this model (2620 > 512). Running this sequence through the model will result in indexing errors


In [70]:
sorted_keys_flausch = sorted(token_dict_flausch, key=token_dict_flausch.get)
sorted_keys_not_flausch = sorted(token_dict_not_flausch, key=token_dict_not_flausch.get)
reverse_keys_flausch = sorted(token_dict_flausch, key=token_dict_flausch.get, reverse=True)
reverse_keys_not_flausch = sorted(token_dict_not_flausch, key=token_dict_not_flausch.get, reverse=True)

In [71]:
def get_indicators(top_perc, lowest_perc):
    # get flausch / not flausch indicator lists for given percentages

    # number of tokens for given percentages
    top_tokens_num = round(len(token_dict_all) * top_perc)
    lowest_tokens_num = round(len(token_dict_all) * lowest_perc)

    # top / lowest values for flausch & not flausch
    value_top_flausch = token_dict_flausch[reverse_keys_flausch[top_tokens_num]]
    value_top_not_flausch = token_dict_not_flausch[reverse_keys_not_flausch[top_tokens_num]]
    value_lowest_flausch = token_dict_flausch[sorted_keys_flausch[lowest_tokens_num]]
    value_lowest_not_flausch = token_dict_not_flausch[sorted_keys_not_flausch[lowest_tokens_num]]

    # top x % flausch
    cutoff_key_top_flausch = list(token_dict_flausch.keys())[list(token_dict_flausch.values()).index(value_top_flausch - 1)]
    top_flausch = reverse_keys_flausch[:reverse_keys_flausch.index(cutoff_key_top_flausch)]

    # top x % not flausch
    cutoff_key_top_not_flausch = list(token_dict_not_flausch.keys())[list(token_dict_not_flausch.values()).index(value_top_not_flausch - 1)]
    top_not_flausch = reverse_keys_not_flausch[:reverse_keys_not_flausch.index(cutoff_key_top_not_flausch)]

    # lowest y % flausch
    cutoff_key_lowest_flausch = list(token_dict_flausch.keys())[list(token_dict_flausch.values()).index(value_lowest_flausch + 1)]
    lowest_flausch = sorted_keys_flausch[:sorted_keys_flausch.index(cutoff_key_lowest_flausch)]

    # lowest y % not flausch
    cutoff_key_lowest_not_flausch = list(token_dict_not_flausch.keys())[list(token_dict_not_flausch.values()).index(value_lowest_not_flausch + 1)]
    lowest_not_flausch = sorted_keys_not_flausch[:sorted_keys_not_flausch.index(cutoff_key_lowest_not_flausch)]

    # indicator lists based on lowest & highest frequency tokens for flausch & not flausch

    flausch_indicators = [x for x in top_flausch if x in lowest_not_flausch]
    not_flausch_indicators = [x for x in top_not_flausch if x in lowest_flausch]

    return [flausch_indicators, not_flausch_indicators]

In [72]:
[flausch_indicators,not_flausch_indicators] = get_indicators(0.5,0.5)

In [73]:
# trying to see if flausch & not flausch indicators seem useful
print([tokenizer.convert_ids_to_tokens(f) for f in flausch_indicators])
print([tokenizer.convert_ids_to_tokens(nf) for nf in not_flausch_indicators])

['Bru', '##46', '##18', 'ansehen', '##hu', '##ons', 'Grie', 'schicken', '##aufen', 'sat', 'Cup', '##ati', 'Max', '##opf', 'Main', '##oto', 'Prozent', 'Köln', 'desto', 'Blö', '##ziehe', 'nieder', '##60', 'rät', '##gesch', 'Tim', '##56', 'aufwend', '##RL', '##igste', 'Aussehen', 'graus', '##sar', '##resse', '##35', 'Kun', 'behauptet', 'kämen', '##liebe', '##BO', '##swert', '##schn', '##icki', 'Chance', '##DL', 'sport', 'Hass', 'fallen', '##uel', 'lag', 'Freude', '##aw', '##äck', '##Ro', '##unen', 'Krieg', '##ont', '##mand', 'Wörter', 'Air', '##inner', '##umpf', 'einzelnen', '##ript', '##hof', 'Zehn', 'Schl', 'wirk', 'akt', 'lief', '##hes', 'holl', '##62', '##üssen', 'Bücher', 'offiziell', '##digen', 'äußern', 'Texten', '##hann', '##legt', '##standen', 'Einen', '##dukt', 'Bour', 'gleichzeitig', '##setzt', '##fan', 'Blan', 'Space', 'sobald', 'treu', 'Spenden', 'Sonntag', '##alit', 'Ph', '##dh', '##ippt', 'besteht', '##66', '##bischen', 'schätzen', '##holen', '##stellt', 'deutschen', 'Mö', 

In [76]:
# looking at flausch & not flausch indicators, simple count

predictions_count_both = []


for i in range(len(test_task1)):

    doc = test_task1.iloc[i]["document"]
    id = test_task1.iloc[i]["comment_id"]
    
    comment = test_task1[(test_task1["document"] == doc) & (test_task1["comment_id"] == id)]["comment"].item()

    tokens = tokenizer.tokenize(comment)
    ids = tokenizer.convert_tokens_to_ids(tokens)

    flausch_score = 0

    for t in ids:
        if t in flausch_indicators:
            flausch_score += 1
        if t in not_flausch_indicators:
            flausch_score -= 1

    # if neutral (score 0), guess not flausch
    guess = "no"
    if flausch_score > 0:
        guess = "yes"
    
    predictions_count_both.append(guess)

pred_count_both = test_task1[["document", "comment_id"]].copy()
pred_count_both["flausch"] = predictions_count_both
pred_count_both.to_csv("predictions/pred_count_both_test_task1.csv", index=False)

In [77]:
binary_eval("Input/Data/train/gold_test_task1.csv", "predictions/pred_count_both_test_task1.csv")

(0.2727272727272727, 0.09482758620689655, 0.14072494669509591)

In [78]:
# looking at flausch indicators, simple count

predictions_count_flausch = []


for i in range(len(test_task1)):

    doc = test_task1.iloc[i]["document"]
    id = test_task1.iloc[i]["comment_id"]
    
    comment = test_task1[(test_task1["document"] == doc) & (test_task1["comment_id"] == id)]["comment"].item()

    tokens = tokenizer.tokenize(comment)
    ids = tokenizer.convert_tokens_to_ids(tokens)

    flausch_score = 0

    for t in ids:
        if t in flausch_indicators:
            flausch_score += 1

    guess = "no"
    if flausch_score > 0:
        guess = "yes"
    
    predictions_count_flausch.append(guess)

pred_count_flausch = test_task1[["document", "comment_id"]].copy()
pred_count_flausch["flausch"] = predictions_count_flausch
pred_count_flausch.to_csv("predictions/pred_count_flausch_test_task1.csv", index=False)

In [79]:
binary_eval("Input/Data/train/gold_test_task1.csv", "predictions/pred_count_flausch_test_task1.csv")

(0.26956521739130435, 0.14846743295019157, 0.19147621988882027)

In [81]:
# looking at not flausch indicators, simple count

predictions_count_not_flausch = []


for i in range(len(test_task1)):

    doc = test_task1.iloc[i]["document"]
    id = test_task1.iloc[i]["comment_id"]
    
    comment = test_task1[(test_task1["document"] == doc) & (test_task1["comment_id"] == id)]["comment"].item()

    tokens = tokenizer.tokenize(comment)
    ids = tokenizer.convert_tokens_to_ids(tokens)

    flausch_score = 0

    for t in ids:
        if t in not_flausch_indicators:
            flausch_score -= 1

    guess = "no"
    if flausch_score == 0:
        guess = "yes"
    
    predictions_count_not_flausch.append(guess)

pred_count_not_flausch = test_task1[["document", "comment_id"]].copy()
pred_count_not_flausch["flausch"] = predictions_count_not_flausch
pred_count_not_flausch.to_csv("predictions/pred_count_not_flausch_test_task1.csv", index=False)

In [ ]:
binary_eval("Input/Data/train/gold_test_task1.csv", "predictions/pred_count_not_flausch_test_task1.csv")

# higher results because more likely to guess flausch

(0.28309305373525556, 0.8275862068965517, 0.421875)

logistic regression with different features

In [83]:
# all caps

def all_caps(comment):

    upper_count = 0

    current = ""
    last_upper = False

    in_upper = False

    for c in comment:

        # idea: notice when at least two uppercase characters in a row -> upper_count up, 
        # but don't continue upping count for every character, only when new word starts (after whitespace)

        current = c
        
        if current.isupper(): # current character is uppercase

            if last_upper: # previous character was uppercase

                if not in_upper: # second character in a row
                    upper_count += 1
                    in_upper = True
            
        else: # current is not uppercase
            in_upper = False # when uppercase character chain ends
        
        last_upper = current.isupper()
    
    return upper_count

In [84]:
# gets count of how many characters are repeated more than needed
# (also detects rare cases where three letters follow each other in normal spelling)
# (does not differentiate between letters & Satzzeichen -> could be solved using regex)

def extra_chars(comment):

    extra_count = 0 # count how many additional characters there are
    rep_count = 0 # count how often a character is repeated

    current = ""
    last = ""
    
    for c in comment:
        current = c

        if current == last: # repeated character
            rep_count += 1

            if rep_count > 1: # character repeated more than once
                extra_count += 1
        
        else:
            rep_count = 0 # currently no repetition
            last = current
    
    return extra_count

In [85]:
X_train = []
y_train = []

for i in range(len(train_task1)):

    doc = train_task1.iloc[i]["document"]
    id = train_task1.iloc[i]["comment_id"]

    flausch = train_task1.iloc[i]["flausch"]
    if flausch == "yes":
        y_train.append(1)
    else:
        y_train.append(0)
    
    comment = train_task1[(train_task1["document"] == doc) & (train_task1["comment_id"] == id)]["comment"].item()

    tokens = tokenizer.tokenize(comment)
    ids = tokenizer.convert_tokens_to_ids(tokens)

    feature_vec = []

    # sentiment
    feature_vec.append(train_task1.iloc[0]["sentiment_orig"])

    # number of flausch & not flausch tokens
    flausch_count = 0
    not_flausch_count = 0

    for t in tokens:
        if t in flausch_indicators:
            flausch_count += 1
        if t in not_flausch_indicators:
            not_flausch_count += 1
    
    feature_vec.append(flausch_count)
    feature_vec.append(not_flausch_count)

    # ratio flausch to all tokens
    feature_vec.append(flausch_count / len(tokens))

    # check presence of top 10 flausch indicators separately
    for f in flausch_indicators[:10]:
        if f in tokens:
            feature_vec.append(1)
        else:
            feature_vec.append(0)

    # number of extra characters
    feature_vec.append(extra_chars(comment))
    # number of words in all uppercase
    feature_vec.append(all_caps(comment))

    X_train.append(feature_vec)

In [ ]:
X_test = []
y_test = []

for i in range(len(test_task1)):

    doc = test_task1.iloc[i]["document"]
    id = test_task1.iloc[i]["comment_id"]

    flausch = test_task1.iloc[i]["flausch"]
    if flausch == "yes":
        y_test.append(1)
    else:
        y_test.append(0)
    
    comment = test_task1[(test_task1["document"] == doc) & (test_task1["comment_id"] == id)]["comment"].item()

    tokens = tokenizer.tokenize(comment)
    ids = tokenizer.convert_tokens_to_ids(tokens)

    feature_vec = []

    # sentiment
    feature_vec.append(test_task1.iloc[0]["sentiment_orig"])

    # number of flausch & not flausch tokens
    flausch_count = 0
    not_flausch_count = 0

    for t in tokens:
        if t in flausch_indicators:
            flausch_count += 1
        if t in not_flausch_indicators:
            not_flausch_count += 1
    
    feature_vec.append(flausch_count)
    feature_vec.append(not_flausch_count)

    # ratio flausch to all tokens
    feature_vec.append(flausch_count / len(tokens))

    # check presence of top 10 flausch indicators separately
    for f in flausch_indicators[:10]:
        if f in tokens:
            feature_vec.append(1)
        else:
            feature_vec.append(0)

    # number of extra characters
    feature_vec.append(extra_chars(comment))
    # number of words in all uppercase
    feature_vec.append(all_caps(comment))

    X_test.append(feature_vec)

In [87]:
log_reg = LogisticRegression().fit(X_train, y_train)

y_pred = log_reg.predict(X_test)

print(classification_report(y_test, y_pred, zero_division=0.0))

              precision    recall  f1-score   support

           0       0.72      1.00      0.84      2662
           1       0.38      0.00      0.01      1044

    accuracy                           0.72      3706
   macro avg       0.55      0.50      0.42      3706
weighted avg       0.62      0.72      0.60      3706



In [93]:
def get_yes_no(x):
    if x == 0:
        return "no"
    else:
        return "yes"

In [96]:
pred_log_reg = test_task1[["document", "comment_id"]].copy()
prediction_log_reg = [get_yes_no(p) for p in y_pred]
pred_log_reg["flausch"] = prediction_log_reg
pred_log_reg.to_csv("predictions/pred_log_reg_test_task1.csv", index=False)

In [97]:
binary_eval("Input/Data/train/gold_test_task1.csv", "predictions/pred_log_reg_test_task1.csv")

(0.375, 0.0028735632183908046, 0.0057034220532319385)